# Healthcare Chatbot

A healthcare chatbot that:

Blocks off-topic or harmful requests

Redacts patient PII (emails, credit card numbers)

Requires human approval before booking appointments

Validates that outputs are medically appropriate

In [23]:

from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage

In [24]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.7
)

In [25]:
# --- Healthcare-specific content filter ---
class HealthcareSafetyFilter(AgentMiddleware):
    """Block non-medical or harmful requests in a healthcare context."""

    BLOCKED_TOPICS = ["drug synthesis", "self-harm", "suicide method", "weapon", "hack"]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_msg = state["messages"][0]
        if first_msg.type != "human":
            return None

        content = first_msg.content.lower()
        for topic in self.BLOCKED_TOPICS:
            if topic in content:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I'm a healthcare assistant and can only help with "
                            "medical questions, appointments, and health information. "
                            "If you're in crisis, please call 112 or your local emergency number."
                        )
                    }],
                    "jump_to": "end"
                }
        return None
    


In [26]:
class MedicalOutputValidator(AgentMiddleware):

    @hook_config(can_jump_to=["end"])
    def after_agent(
        self,
        state: AgentState,
        runtime: Runtime
    ) -> dict[str, Any] | None:

        if not state["messages"]:
            return None

        last_message = state["messages"][-1]

        if not isinstance(last_message, AIMessage):
            return None

        content = last_message.content

        # Convert list content into string
        if isinstance(content, list):
            text_parts = []

            for item in content:
                if isinstance(item, dict) and "text" in item:
                    text_parts.append(item["text"])

            content = "".join(text_parts)

        content = content.strip()

        # Now string operations work
        if "doctor" not in content.lower():
            content += (
                "\n\nDisclaimer: This information is for general "
                "educational purposes and is not a substitute for "
                "professional medical advice. Please consult a "
                "qualified healthcare professional for diagnosis "
                "or treatment."
            )

        last_message.content = content

        return None

In [27]:
# --- Healthcare tools ---
@tool
def search_symptoms(symptoms: str) -> str:
    """Search for information about medical symptoms."""
    return f"Symptom information for: {symptoms}. Please consult a doctor for diagnosis."

@tool
def book_appointment(patient_name: str, date: str, doctor: str) -> str:
    """Book a medical appointment."""
    return f"Appointment booked for {patient_name} with Dr. {doctor} on {date}"

@tool
def get_medication_info(medication: str) -> str:
    """Get information about a medication."""
    return f"General info about {medication}. Always follow your doctor's prescription."



In [28]:
healthcare_bot = create_agent(
    model=llm,
    tools=[search_symptoms, book_appointment, get_medication_info],
    middleware=[
        # Guardrail 1: Block harmful/off-topic requests
        HealthcareSafetyFilter(),

        # Guardrail 2: Redact patient PII from inputs
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Guardrail 3: Require approval before booking appointments
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "search_symptoms": False,
                "get_medication_info": False,
            }
        ),

        # Guardrail 4: Add medical disclaimer to all outputs
        MedicalOutputValidator(),
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis."
    )
)

print("🏥 Healthcare chatbot with full guardrail stack created!")

🏥 Healthcare chatbot with full guardrail stack created!


In [29]:
# Test 1: Safe medical query
config_t1 = {"configurable": {"thread_id": "healthcare_session_t1"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "What are symptoms of Type 2 Diabetes?"}]},
    config=config_t1
)

print(result["messages"][-1].content)

e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Common symptoms of Type 2 Diabetes can include:

*   Increased thirst (polydipsia)
*   Frequent urination (polyuria)
*   Increased hunger (polyphagia)
*   Unexplained weight loss
*   Fatigue and weakness
*   Blurred vision
*   Slow-healing sores or frequent infections
*   Areas of darkened skin (acanthosis nigricans), often on the neck or armpits

Please remember that these symptoms can be associated with other conditions, and it is very important to consult a qualified healthcare professional for a proper diagnosis and personalized medical advice. 

Would you like help booking an appointment with a doctor?


In [30]:

# Test 2: Query with PII (email gets redacted)
result = healthcare_bot.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is patient123@gmail.com. What can I take for a headache?"
    }]},
    config=config_t1
)
print("=== PII Redaction Test ===")
print(result["messages"][-1].content)

e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== PII Redaction Test ===
For common tension headaches, over-the-counter pain relievers such as **ibuprofen**, **acetaminophen**, or **aspirin** are often used. 

However, before taking any medication, it's important to consider:
*   Are you taking any other medications or do you have any underlying health conditions?
*   How severe is the headache, and how long has it lasted?

*(Note: I've noted your email address, but please remember I cannot prescribe medications or provide a definitive treatment plan without knowing more about your health history.)*

If headaches are frequent, severe, or accompanied by symptoms like dizziness, vision changes, or confusion, you should definitely consult a doctor. Would you like me to help you book an appointment with a healthcare provider?


In [31]:

# Test 3: Off-topic / harmful request — gets blocked
result = healthcare_bot.invoke({
    "messages": [{"role": "user", "content": "How do I synthesize drugs at home?"}]
},
 config=config_t1)
print("=== Blocked Request ===")
print(result["messages"][-1].content)

e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== Blocked Request ===
I cannot provide instructions on how to synthesize drugs at home. 

If you or someone you know is struggling with substance use or looking for help related to medications, support is available. You can reach out to a healthcare professional, or contact resources like the Substance Abuse and Mental Health Services Administration (SAMHSA) National Helpline at 1-800-662-4357 for free, confidential, 24/7 support. 

Let me know if you need help finding medical care, booking an appointment, or learning about safe, prescribed treatments.

Disclaimer: This information is for general educational purposes and is not a substitute for professional medical advice. Please consult a qualified healthcare professional for diagnosis or treatment.


In [32]:
#  Test 4: Appointment booking — requires human approval
config = {"configurable": {"thread_id": "healthcare_session_001"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "Book me an appointment with Dr. Sharma on Sep 15"}]},
    config=config
)
print("=== Appointment Booking — Awaiting Approval ===")
print(result)



e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== Appointment Booking — Awaiting Approval ===
{'messages': [HumanMessage(content='Book me an appointment with Dr. Sharma on Sep 15', additional_kwargs={}, response_metadata={}, id='67909453-11ea-4818-97ef-47d71b0bf771'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'book_appointment', 'arguments': '{"date": "Sep 15", "patient_name": "[Patient Name]", "doctor": "Dr. Sharma"}'}, '__gemini_function_call_thought_signatures__': {'call_483391': 'El4KXAERTTIPacrerbfdqbAURpdERMaAOn06ldXoJc5Kv9paK8YBdwepy6O/XKi2gqLpEnjMp3btnaFcSyOZBqYmx+3sT/B1JkE1+b4DdX0+Tc/9rLfAk7PtTm+Gu6pJ'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a05eac-3340-7861-bb83-4078f2cc8908-0', tool_calls=[{'name': 'book_appointment', 'args': {'date': 'Sep 15', 'patient_name': '[Patient Name]', 'doctor': 'Dr. Sharma'}, 'id': 'call_483391', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

In [33]:
# Approve
from langgraph.types import Command
approved = healthcare_bot.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)
print("\n=== After Approval ===")
print(approved["messages"][-1].content)

e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



=== After Approval ===
I've gone ahead and booked an appointment for you with Dr. Sharma on Sep 15. Please remember to arrive a little early and bring any relevant medical information you might need.

Disclaimer: This information is for general educational purposes and is not a substitute for professional medical advice. Please consult a qualified healthcare professional for diagnosis or treatment.
